In [1]:
!ls -la /kaggle/input/datasets/abhinavharbola/recsys-movielens-code/kaggle_upload/

total 36396
drwxr-xr-x 4 nobody nogroup        0 Aug 13 12:14 .
drwxr-xr-x 3 nobody nogroup        0 Aug 13 12:14 ..
drwxr-xr-x 2 nobody nogroup        0 Aug 13 12:14 scripts
drwxr-xr-x 8 nobody nogroup        0 Aug 13 12:14 src
-rw-r--r-- 1 nobody nogroup 37269299 Aug 13 12:15 train.parquet


In [2]:
!nvidia-smi

Thu Aug 13 12:17:52 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   52C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
!pip install polars pyarrow mlflow -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 3.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 89.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 84.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 73.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.0/212.0 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.9/123.9 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 53.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 214.9/214.9 kB 15.9 MB/s eta 0:00:00


In [4]:
SRC_ROOT = "/kaggle/input/datasets/abhinavharbola/recsys-movielens-code/kaggle_upload"

!cp -r {SRC_ROOT}/src /kaggle/working/
!cp -r {SRC_ROOT}/scripts /kaggle/working/
!mkdir -p /kaggle/working/data/processed /kaggle/working/checkpoints
!cp {SRC_ROOT}/train.parquet /kaggle/working/data/processed/
!ls /kaggle/working/data/processed/

train.parquet


In [5]:
%cd /kaggle/working

!python scripts/train_two_tower.py \
    --train-path data/processed/train.parquet \
    --checkpoint-path checkpoints/two_tower.pt \
    --output-dir data/processed \
    --epochs 10

/kaggle/working
epoch 0: avg_loss=6.1332
epoch 1: avg_loss=5.7853
epoch 2: avg_loss=5.6338
epoch 3: avg_loss=5.5631
epoch 4: avg_loss=5.5097
epoch 5: avg_loss=5.4666
epoch 6: avg_loss=5.4237
epoch 7: avg_loss=5.3758
epoch 8: avg_loss=5.3434
epoch 9: avg_loss=5.3231
2026/08/13 12:51:27 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/08/13 12:51:28 INFO mlflow.store.db.utils: Updating database tables
2026/08/13 12:51:30 INFO mlflow.tracking.fluent: Experiment with name 'movielens-recsys-benchmark' does not exist. Creating a new experiment.
exported 148745 user and 31195 item embeddings to data/processed


In [6]:
!python scripts/train_sasrec.py \
    --train-path data/processed/train.parquet \
    --checkpoint-path checkpoints/sasrec.pt \
    --output-dir data/processed \
    --epochs 10

/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:431: UserWarning: Support for mismatched src_key_padding_mask and mask is deprecated. Use same type for both instead.
  src_key_padding_mask = F._canonical_mask(
epoch 0: avg_loss=9.9140
epoch 1: avg_loss=7.6694
epoch 2: avg_loss=7.4639
epoch 3: avg_loss=7.2168
epoch 4: avg_loss=7.0495
epoch 5: avg_loss=6.9689
epoch 6: avg_loss=6.8815
epoch 7: avg_loss=6.7893
epoch 8: avg_loss=6.7275
epoch 9: avg_loss=6.6820
/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:431: UserWarning: Support for mismatched src_key_padding_mask and mask is deprecated. Use same type for both instead.
  src_key_padding_mask = F._canonical_mask(
exported 148745 user and 31195 item embeddings to data/processed


In [7]:
!ls /kaggle/working/data/processed/

sasrec_item_embeddings.parquet	two_tower_item_embeddings.parquet
sasrec_user_embeddings.parquet	two_tower_user_embeddings.parquet
train.parquet
